# Data Vector and Gaussian Covariance

This notebook builds a cosmological data vector from multiple tracer pairs
with per-pair ℓ ranges and bandpower binning, then computes the analytic
Gaussian covariance matrix.

### Steps
1. Define tracers with survey noise parameters
2. Build a data vector with per-pair ℓ cuts and log-binning
3. Compute the Gaussian covariance matrix
4. Visualize the data vector, covariance, correlation matrix, and S/N

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

## 1. Define cosmology and tracers with noise

In [ ]:
PLANCK = dict(
    Omega_c=0.2589, Omega_b=0.0486, h=0.6774,
    n_s=0.9667, sigma8=0.8159, Omega_k=0.0, w0=-1.0, wa=0.0,
)

z = np.linspace(0.0, 3.0, 200)
nz_source = np.exp(-0.5 * ((z - 0.8) / 0.2)**2)
nz_lens = np.exp(-0.5 * ((z - 0.4) / 0.15)**2)

TRACER_CONFIGS = {
    "wl": dict(
        tracer_type="weak_lensing",
        dndz_z=z.tolist(), dndz_nz=nz_source.tolist(),
        has_shear=True,
        n_gal=10.0,         # 10 gal/arcmin^2
        sigma_epsilon=0.26, # shape noise
    ),
    "nc": dict(
        tracer_type="number_counts",
        dndz_z=z.tolist(), dndz_nz=nz_lens.tolist(),
        bias_model="linear", bias_params={"a0": 1.5, "a1": 0.3},
        has_rsd=False,
        n_gal=30.0,         # 30 gal/arcmin^2
    ),
}

print("Tracers:")
for name, cfg in TRACER_CONFIGS.items():
    noise = f"n_gal={cfg['n_gal']}"
    if cfg.get("sigma_epsilon"):
        noise += f", σ_ε={cfg['sigma_epsilon']}"
    print(f"  {name}: {cfg['tracer_type']}, {noise}")

## 2. Configure the data vector

Different ℓ ranges per pair (WL to higher ℓ, NC to lower ℓ),
with logarithmic bandpower binning.

In [ ]:
from tissage_cosmique.computations.ell_binning import make_log_bins, pair_key

# Shared fine ell grid for computation
ell_fine = np.arange(10, 2001).astype(float)

# Per-pair ell ranges (different scale cuts)
ell_wl_wl = np.arange(30, 2001).astype(float)
ell_nc_nc = np.arange(20, 501).astype(float)
ell_wl_nc = np.arange(30, 1001).astype(float)

# Log-spaced bins per pair
bins_wl_wl = make_log_bins(30, 2000, 15)
bins_nc_nc = make_log_bins(20, 500, 10)
bins_wl_nc = make_log_bins(30, 1000, 12)

dv_config = {
    "tracer_names": ["wl", "nc"],
    "ell_per_pair": {
        "wl-wl": ell_wl_wl.tolist(),
        "wl-nc": ell_wl_nc.tolist(),
        "nc-nc": ell_nc_nc.tolist(),
    },
    "bin_edges_per_pair": {
        "wl-wl": bins_wl_wl.tolist(),
        "wl-nc": bins_wl_nc.tolist(),
        "nc-nc": bins_nc_nc.tolist(),
    },
}

n_bins_total = 15 + 12 + 10
print(f"Data vector: 3 pairs, {n_bins_total} total bins")
print(f"  WL×WL: ℓ ∈ [30, 2000], {15} bins")
print(f"  WL×NC: ℓ ∈ [30, 1000], {12} bins")
print(f"  NC×NC: ℓ ∈ [20, 500],  {10} bins")

## 3. Compute data vector and covariance

In [ ]:
from tissage_cosmique.computations.covariance import compute_covariance_from_data_vector
from tissage_cosmique.computations.data_vector import make_data_vector_result

f_sky = 0.3  # ~12,000 deg^2 (e.g., Rubin LSST)

dv, dv_result, cov = compute_covariance_from_data_vector(
    PLANCK, TRACER_CONFIGS, dv_config, ell_fine, f_sky,
)

print(f"Data vector length: {len(dv)}")
print(f"Covariance shape:   {cov.shape}")
print(f"Pairs: {dv_result.pairs}")
print(f"Segment lengths: {dv_result.lengths}")

## 4. Visualize the data vector

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

pair_labels = {"wl-wl": "WL × WL", "wl-nc": "WL × NC", "nc-nc": "NC × NC"}
pair_colors = {"wl-wl": "tab:blue", "wl-nc": "tab:orange", "nc-nc": "tab:green"}

for ax, (t1, t2) in zip(axes, dv_result.pairs):
    key = pair_key(t1, t2)
    cl = dv_result.get_cl(t1, t2)
    ell_eff = dv_result.get_ell(t1, t2)
    err = np.sqrt(np.diag(cov)[
        dv_result.offsets[dv_result.pairs.index((t1, t2))]:
        dv_result.offsets[dv_result.pairs.index((t1, t2))] + len(cl)
    ])

    ax.errorbar(ell_eff, ell_eff * (ell_eff + 1) * np.abs(cl) / (2 * np.pi),
                yerr=ell_eff * (ell_eff + 1) * err / (2 * np.pi),
                fmt="o", ms=4, color=pair_colors[key], capsize=2)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$\ell(\ell+1) C_\ell / 2\pi$")
    ax.set_title(f"{pair_labels[key]}")

fig.suptitle(f"Binned data vector with 1σ errors (f_sky={f_sky})", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Covariance and correlation matrices

In [ ]:
# Correlation matrix
diag = np.sqrt(np.diag(cov))
corr = cov / np.outer(diag, diag)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Covariance (log scale)
ax = axes[0]
vmax = np.max(np.abs(cov))
im = ax.imshow(np.abs(cov), norm=LogNorm(vmin=vmax * 1e-8, vmax=vmax), cmap="viridis")
plt.colorbar(im, ax=ax, label="|Cov|")

# Mark pair boundaries
for off in dv_result.offsets[1:]:
    ax.axhline(off - 0.5, color="white", lw=0.5, ls="--")
    ax.axvline(off - 0.5, color="white", lw=0.5, ls="--")

ax.set_title("Covariance matrix")
ax.set_xlabel("Data vector index")
ax.set_ylabel("Data vector index")

# Correlation
ax = axes[1]
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
plt.colorbar(im, ax=ax, label="Correlation")

for off in dv_result.offsets[1:]:
    ax.axhline(off - 0.5, color="black", lw=0.5, ls="--")
    ax.axvline(off - 0.5, color="black", lw=0.5, ls="--")

ax.set_title("Correlation matrix")
ax.set_xlabel("Data vector index")
ax.set_ylabel("Data vector index")

# Add pair labels
for ax in axes:
    for i, ((t1, t2), off, length) in enumerate(
        zip(dv_result.pairs, dv_result.offsets, dv_result.lengths)
    ):
        mid = off + length / 2
        ax.text(mid, -1.5, pair_labels[pair_key(t1, t2)],
                ha="center", fontsize=8, color="black")

plt.tight_layout()
plt.show()

## 6. Signal-to-noise ratio

In [ ]:
# Per-bin S/N
snr_per_bin = np.abs(dv) / np.sqrt(np.diag(cov))

# Total S/N (Fisher forecast)
cov_inv = np.linalg.inv(cov)
total_snr = np.sqrt(dv @ cov_inv @ dv)

print(f"Total S/N = {total_snr:.1f}")

fig, ax = plt.subplots(figsize=(10, 4))

for i, (t1, t2) in enumerate(dv_result.pairs):
    key = pair_key(t1, t2)
    off = dv_result.offsets[i]
    n = dv_result.lengths[i]
    ell_eff = dv_result.get_ell(t1, t2)
    ax.plot(ell_eff, snr_per_bin[off:off + n], "o-",
            color=pair_colors[key], ms=5, label=pair_labels[key])

ax.set_xlabel(r"$\ell_{\rm eff}$")
ax.set_ylabel("S/N per bin")
ax.set_title(f"Signal-to-noise per bin (total S/N = {total_snr:.1f})")
ax.set_xscale("log")
ax.legend()
ax.axhline(1, color="gray", ls="--", lw=0.5)
plt.tight_layout()
plt.show()

## 7. Comparison: varying f_sky

In [ ]:
fsky_values = [0.05, 0.1, 0.3, 0.5]
total_snrs = []

for fs in fsky_values:
    _, _, c = compute_covariance_from_data_vector(
        PLANCK, TRACER_CONFIGS, dv_config, ell_fine, fs,
    )
    ci = np.linalg.inv(c)
    total_snrs.append(np.sqrt(dv @ ci @ dv))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fsky_values, total_snrs, "bo-", ms=8)
ax.set_xlabel("$f_{\\rm sky}$")
ax.set_ylabel("Total S/N")
ax.set_title("Total signal-to-noise vs sky fraction")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for fs, snr in zip(fsky_values, total_snrs):
    print(f"  f_sky={fs:.2f}: S/N = {snr:.1f}")

## Summary

This notebook demonstrated:

- **Per-pair ℓ ranges**: WL×WL to ℓ=2000, NC×NC to ℓ=500 (different scale cuts)
- **Bandpower binning**: log-spaced bins per pair, reducing from ~2000 multipoles to ~37 data points
- **Gaussian covariance**: analytic formula with shape noise (WL) and shot noise (NC)
- **Correlation matrix**: shows cross-spectrum correlations between pairs
- **S/N analysis**: per-bin and total signal-to-noise, scaling with f_sky

The data vector + covariance are the inputs to a cosmological likelihood:
```
χ² = (d - m(θ))ᵀ C⁻¹ (d - m(θ))
```
where `d` is the observed data vector, `m(θ)` is the model (emulated), and
`C` is this covariance matrix.